# TripMe Sri Lanka — OpenStreetMap Places Collector

**Goal:** grow the places dataset from 2278 towards ~5000 by pulling additional
Sri Lankan points of interest from **OpenStreetMap** (ODbL license, attribution
required: "© OpenStreetMap contributors") via the free **Overpass API**.

**One `.jsonl` file per category** — matching the existing `data/raw/` layout
(`tripme_database_complete_beach.json`, `..._water-falls.json`, etc: one file
per category, not one combined dump). Each category writes its own file, e.g.
`osm_buddhist_temple.jsonl`, `osm_cascade.jsonl`, `osm_sandy_beach.jsonl`, ...

**How it avoids duplicating the existing 2278 places:**
- Queries OSM **per district**, so every fetched place already gets a real `district_id`
  (no reverse-geocoding / boundary polygon file needed).
- Deduplicates against `data/processed/places.json` by name-similarity + proximity
  (within ~300m of an existing place with a similar name = skip).

**What OSM does NOT give us**, so this notebook fills in:
- `description` — generated by a small **local LLM** (Qwen2.5-3B-Instruct,
  4-bit quantized, runs free on a Kaggle T4 GPU - no paid API calls) grounded
  strictly in the facts OSM actually gave us (name, category, district). Falls
  back to a rule-based template for any record the model fails on, so the
  step never blocks the pipeline.
- Visitor-facing fields (`opening_hours`, `budget_category`, `safety_level`, ...) —
  filled with sensible category-based defaults. Easy to spot for hand-review later
  since each record's `id` is prefixed `pl_osm_`.

**Output:** one pretty-printed `.jsonl` file per category (same multi-line-per-
record style as `data/raw/place_data.jsonl`). Download all of them and drop them
into `data/raw/`, then re-run `scripts/01_merge_places.py`.

**GPU:** enable it (Settings → Accelerator → T4 x2 or similar) — needed for the
local LLM description step. Everything else in the notebook is just HTTP calls.

**Resumability:** both the slow steps checkpoint to disk as they go
(`osm_raw_elements.json` for the Overpass fetch loop, `osm_candidate_records.jsonl`
for the per-place LLM description step) - re-running after a disconnect picks up
where it left off instead of starting over. This only survives a **restarted
session on the same Kaggle notebook instance**; `/kaggle/working` is wiped if you
start a brand new session, so don't delete/recreate the notebook mid-run.

**Rough total runtime:** ~15-30 min for the Overpass fetch stage (500 queries,
network-bound) + ~1-3 sec per place for LLM descriptions (so total time scales
with however many places OSM actually returns - could be anywhere from 30 min
to a couple hours). Comfortably fits Kaggle's free 9-hour session / 30 GPU-hour
weekly limits either way.


## 1. Setup

In [ ]:
!pip install -q -U requests tqdm transformers accelerate bitsandbytes sentencepiece
print("Dependencies installed.")

In [ ]:
import json
import random
import re
import time
import uuid
from pathlib import Path

import requests
from tqdm.auto import tqdm

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
PROGRESS_FILE = OUTPUT_DIR / "osm_collect_progress.json"
RAW_ELEMENTS_FILE = OUTPUT_DIR / "osm_raw_elements.json"

# If you have the existing merged dataset available (upload places.json as a
# Kaggle/Colab dataset, or leave as None to skip dedup against it and dedupe
# only within this run).
EXISTING_PLACES_PATH = None  # e.g. Path("/kaggle/input/tripme-places/places.json")

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
REQUEST_DELAY = 2.0  # seconds between Overpass calls - be polite to the free public instance
REQUEST_TIMEOUT = 90

# The public Overpass instance returns 406 Not Acceptable for requests.post's
# default User-Agent ("python-requests/x.x") - it's filtered as unidentified
# bot traffic. A descriptive User-Agent (Overpass's own docs ask for one that
# identifies the tool) fixes it.
HTTP_SESSION = requests.Session()
HTTP_SESSION.headers.update({
    "User-Agent": "TripMeSriLankaPlacesCollector/1.0 (Kaggle notebook; contact: n/a)",
})

print("Output directory:", OUTPUT_DIR)

## 2. Districts & provinces

Matches the canonical spelling already used in `data/processed/places.json`
(after `01_merge_places.py`'s `DISTRICT_ALIASES` normalization), so newly
collected records merge in cleanly without creating new duplicate district names.

In [ ]:
DISTRICT_PROVINCE = {
    "Colombo": "Western", "Gampaha": "Western", "Kalutara": "Western",
    "Kandy": "Central", "Matale": "Central", "Nuwara Eliya": "Central",
    "Galle": "Southern", "Matara": "Southern", "Hambanthota": "Southern",
    "Jaffna": "Northern", "Kilinochchi": "Northern", "Mannar": "Northern",
    "Vavuniya": "Northern", "Mullaitivu": "Northern",
    "Batticaloa": "Eastern", "Ampara": "Eastern", "Trincomalee": "Eastern",
    "Kurunagala": "North Western", "Puttalam": "North Western",
    "Anuradhapura": "North Central", "Polonnaruwa": "North Central",
    "Badulla": "Uva", "Monaragala": "Uva",
    "Ratnapura": "Sabaragamuwa", "Kegalle": "Sabaragamuwa",
}

# OSM district-name spelling sometimes differs slightly from ours - map to canonical.
OSM_DISTRICT_ALIASES = {
    "Hambantota": "Hambanthota",
    "Kurunegala": "Kurunagala",
    "Moneragala": "Monaragala",
    "Nuwara-Eliya": "Nuwara Eliya",
    "NuwaraEliya": "Nuwara Eliya",
}

print(f"{len(DISTRICT_PROVINCE)} districts configured.")

## 3. Category mapping

Maps OSM tag combinations to the `category_id` values already used in the
dataset, and to the output filename for that category. Order matters - more
specific filters are listed first so a place isn't miscategorized as "Other".

In [ ]:
# (overpass tag filter, category_id, activities hint, output filename)
OSM_CATEGORIES = [
    ('["amenity"="place_of_worship"]["religion"="buddhist"]', "Buddhist Temple", "meditation, photography", "osm_buddhist_temple.jsonl"),
    ('["amenity"="place_of_worship"]["religion"="hindu"]', "Kovil", "prayer, photography", "osm_kovil.jsonl"),
    ('["amenity"="place_of_worship"]["religion"="christian"]', "Church", "prayer, photography", "osm_church.jsonl"),
    ('["amenity"="place_of_worship"]["religion"="muslim"]', "Mosque", "prayer, photography", "osm_mosque.jsonl"),
    ('["amenity"="place_of_worship"]', "Temple", "prayer, photography", "osm_temple.jsonl"),
    ('["historic"="archaeological_site"]', "Ruins", "exploring, photography", "osm_ruins.jsonl"),
    ('["historic"="ruins"]', "Ruins", "exploring, photography", "osm_ruins.jsonl"),
    ('["historic"="monument"]', "Building", "photography, sightseeing", "osm_building.jsonl"),
    ('["historic"="memorial"]', "Building", "photography, sightseeing", "osm_building.jsonl"),
    ('["natural"="waterfall"]', "Cascade", "photography, swimming", "osm_waterfall.jsonl"),
    ('["natural"="beach"]', "Sandy Beach", "swimming, sunbathing", "osm_beach.jsonl"),
    ('["leisure"="beach_resort"]', "Sandy Beach", "swimming, relaxing", "osm_beach.jsonl"),
    ('["boundary"="national_park"]', "National Park", "wildlife watching, safari", "osm_national_park.jsonl"),
    ('["leisure"="nature_reserve"]', "National Park", "wildlife watching, hiking", "osm_national_park.jsonl"),
    ('["tourism"="viewpoint"]', "Viewpoint", "photography, sightseeing", "osm_viewpoint.jsonl"),
    ('["tourism"="museum"]', "Museum", "sightseeing, learning", "osm_museum.jsonl"),
    ('["tourism"="zoo"]', "Adventure Park", "wildlife watching", "osm_adventure_park.jsonl"),
    ('["tourism"="theme_park"]', "Adventure Park", "rides, family fun", "osm_adventure_park.jsonl"),
    ('["landuse"="farmland"]["crop"="tea"]', "Tea Estate", "walking, photography", "osm_tea_estate.jsonl"),
    ('["tourism"="attraction"]', "Other", "sightseeing", "osm_other.jsonl"),
]

OUTPUT_FILENAMES = sorted({c[3] for c in OSM_CATEGORIES})
print(f"{len(OSM_CATEGORIES)} OSM tag mappings configured, "
      f"writing to {len(OUTPUT_FILENAMES)} separate category files:")
for fn in OUTPUT_FILENAMES:
    print(" -", fn)

## 4. Overpass query builder & fetcher

In [ ]:
def build_query(district, tag_filter):
    # OSM's admin boundary relation for a Sri Lankan district is named
    # "<District> District" (confirmed via Nominatim for every district in
    # DISTRICT_PROVINCE, e.g. "Kandy District", "Jaffna District") - a plain
    # area["name"="Kandy"] does NOT match the district boundary at all, it
    # matches unrelated features that happen to share the bare name (e.g. a
    # train station literally named "Kandy"), silently returning 0 real
    # results for every query. This was found by testing the exact query
    # against the live Overpass API and comparing to a Sri-Lanka-wide query,
    # which did return real elements.
    area_name = f"{district} District"
    return f"""
    [out:json][timeout:{REQUEST_TIMEOUT}];
    area["name"="Sri Lanka"]->.country;
    area["name"="{area_name}"](area.country)->.searchArea;
    (
      node{tag_filter}(area.searchArea);
      way{tag_filter}(area.searchArea);
    );
    out center tags 200;
    """


def overpass_fetch(query, retries=3):
    for attempt in range(retries):
        try:
            r = HTTP_SESSION.post(OVERPASS_URL, data={"data": query}, timeout=REQUEST_TIMEOUT)
            if r.status_code == 200:
                try:
                    return r.json().get("elements", [])
                except ValueError:
                    # Overpass's free public instance occasionally returns an
                    # HTML error/rate-limit page with a 200 status under load.
                    print(f"  non-JSON 200 response (attempt {attempt+1}), retrying...")
                    time.sleep(10 * (attempt + 1))
                    continue
            if r.status_code == 429 or r.status_code == 504:
                time.sleep(10 * (attempt + 1))
                continue
            if r.status_code == 406:
                # Same root cause as the module-level HTTP_SESSION User-Agent
                # fix, but printed here too in case someone removes/edits the
                # session headers later - 406 from Overpass almost always
                # means the request looks like unidentified bot traffic.
                print(f"  406 Not Acceptable (attempt {attempt+1}) - check HTTP_SESSION's "
                      "User-Agent header is still set, retrying...")
                time.sleep(5 * (attempt + 1))
                continue
            if r.status_code == 403:
                # Seen from Kaggle's shared IP pool even when the same query
                # works fine from other networks - likely an IP-range block
                # or temporary rate-limit on the free public instance rather
                # than anything wrong with the request itself. Retrying with
                # backoff is the only real option against this endpoint; if
                # it persists across all retries, consider switching
                # OVERPASS_URL to a mirror (e.g. https://overpass.osm.ch/api/interpreter -
                # verify with a test query first, mirrors can have stale/
                # incomplete data).
                print(f"  403 Forbidden (attempt {attempt+1}) - possible IP-based block, retrying...")
                time.sleep(15 * (attempt + 1))
                continue
            r.raise_for_status()
        except requests.RequestException as e:
            print(f"  request failed (attempt {attempt+1}): {e}")
            time.sleep(5 * (attempt + 1))
    print("  giving up on this query after retries - returning 0 elements for it.")
    return []


print("Overpass helpers ready.")

## 5. Resumable progress state

Tracks which (district, category) pairs have already been fetched, so a
disconnect/timeout doesn't lose earlier work - re-running the notebook picks
up where it left off.

In [ ]:
def load_progress():
    if PROGRESS_FILE.exists():
        return json.loads(PROGRESS_FILE.read_text(encoding="utf-8"))
    return {"completed": []}


def save_progress(progress):
    PROGRESS_FILE.write_text(json.dumps(progress, indent=2), encoding="utf-8")


progress = load_progress()
print(f"Resuming: {len(progress['completed'])} (district, category) pairs already done.")

## 6. Main fetch loop

Iterates every district x category combination, appending raw OSM elements to
`raw_osm_elements` (kept with their `district`/`category_id`/`output_file` tags
so later cells don't need to re-derive them).

In [ ]:
raw_osm_elements = []
if RAW_ELEMENTS_FILE.exists():
    raw_osm_elements = json.loads(RAW_ELEMENTS_FILE.read_text(encoding="utf-8"))

pairs = [(d, cat, filt, act, fn) for d in DISTRICT_PROVINCE for (filt, cat, act, fn) in OSM_CATEGORIES]

failed_pairs = []
pbar = tqdm(pairs, desc="Fetching OSM places")
for district, category_id, tag_filter, activities, output_file in pbar:
    key = f"{district}::{category_id}::{tag_filter}"
    pbar.set_postfix_str(f"{district[:12]} / {category_id[:15]}")
    if key in progress["completed"]:
        continue
    try:
        query = build_query(district, tag_filter)
        elements = overpass_fetch(query)
    except Exception as e:
        # A single bad (district, category) pair should never take down the
        # whole 500-query run - log it, mark it done (so it isn't retried
        # forever), and move on. Check failed_pairs after the run if the
        # final count looks lower than expected.
        print(f"  UNEXPECTED error for {key}: {e} - skipping this pair.")
        failed_pairs.append(key)
        elements = []
    for el in elements:
        el["_district"] = district
        el["_category_id"] = category_id
        el["_activities"] = activities
        el["_output_file"] = output_file
        raw_osm_elements.append(el)
    progress["completed"].append(key)
    RAW_ELEMENTS_FILE.write_text(json.dumps(raw_osm_elements), encoding="utf-8")
    save_progress(progress)
    time.sleep(REQUEST_DELAY)

print(f"\nFetched {len(raw_osm_elements)} raw OSM elements across "
      f"{len(DISTRICT_PROVINCE)} districts x {len(OSM_CATEGORIES)} category filters.")
if failed_pairs:
    print(f"{len(failed_pairs)} (district, category) pairs hit an unexpected error "
          f"and returned 0 elements: {failed_pairs}")

## 7. Load existing places (for deduplication)

In [ ]:
def norm_name(name):
    return re.sub(r"[^a-z0-9]+", "", (name or "").lower())


existing_places = []
if EXISTING_PLACES_PATH:
    if EXISTING_PLACES_PATH.exists():
        try:
            existing_places = json.loads(EXISTING_PLACES_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError as e:
            print(f"WARNING: {EXISTING_PLACES_PATH} is not valid JSON ({e}) - "
                  "continuing with dedup against existing places DISABLED.")
    else:
        print(f"WARNING: EXISTING_PLACES_PATH is set but {EXISTING_PLACES_PATH} "
              "does not exist (check the dataset is attached via 'Add Input') - "
              "continuing with dedup against existing places DISABLED.")

existing_by_norm_name = {}
for p in existing_places:
    existing_by_norm_name.setdefault(norm_name(p["name"]), []).append(p)

print(f"Loaded {len(existing_places)} existing places for dedup "
      f"({'skipping - none provided' if not existing_places else 'active'}).")

## 8. Load the local LLM for description generation

Same pattern as `Sri_Lanka_History_Dataset_Collector_Automated.ipynb`'s model
step: a small instruct model, 4-bit quantized so it fits comfortably on a
single Kaggle T4 (16GB) alongside everything else running in the notebook.
`Qwen2.5-3B-Instruct` (not the 7B used for chronicle extraction) is enough for
short, fact-constrained one-paragraph descriptions and leaves more headroom.

If model loading fails (no GPU, out of memory, etc.), `llm_ready` stays
`False` and every description falls back to the rule-based template from
Section 9 automatically - this cell is not a hard dependency for the rest of
the notebook to run.

In [ ]:
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

llm_ready = False
llm_tokenizer = None
llm_model = None

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if torch.cuda.is_available():
        print(f"Loading {LLM_MODEL_NAME} (4-bit)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
        llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
        )
        llm_model.eval()
        llm_ready = True
        print("LLM loaded and ready.")
    else:
        print("No GPU detected - skipping LLM load. Descriptions will use the "
              "rule-based template fallback (Section 9). Enable a GPU accelerator "
              "(Settings -> Accelerator) to use the LLM instead.")
except Exception as e:
    print(f"LLM load failed ({e}) - falling back to rule-based templates for all descriptions.")
    llm_ready = False

## 9. Fallback description templates

Rule-based, no LLM required. Used only if the LLM (loaded next) fails to load,
or fails to produce a valid description for a specific record after retries -
so a model hiccup never blocks the whole pipeline. Produces natural-reading
sentences in the same voice as the existing dataset ("X is a serene/historic/...
site located in district, offering ..."), grounded only in facts we actually
have (name, category, district) - never invents history or details OSM didn't
provide.

In [ ]:
CATEGORY_BLURB = {
    "Buddhist Temple": "a Buddhist temple",
    "Kovil": "a Hindu kovil",
    "Church": "a church",
    "Mosque": "a mosque",
    "Temple": "a place of worship",
    "Ruins": "an archaeological site",
    "Building": "a historic landmark",
    "Cascade": "a waterfall",
    "Sandy Beach": "a beach",
    "National Park": "a nature reserve",
    "Viewpoint": "a scenic viewpoint",
    "Museum": "a museum",
    "Adventure Park": "an attraction",
    "Tea Estate": "a tea estate",
    "Other": "a point of interest",
}

OPENER_TEMPLATES = [
    "{name} is {blurb} located in {district}, Sri Lanka.",
    "{name} is {blurb} in the {district} area of Sri Lanka.",
    "Located in {district}, {name} is {blurb} worth visiting.",
]

CLOSER_TEMPLATES = {
    "Buddhist Temple": "Visitors can explore the temple grounds and take part in quiet reflection.",
    "Kovil": "Visitors are welcome to observe the shrine's architecture and rituals respectfully.",
    "Church": "The site offers a peaceful stop for visitors interested in local religious heritage.",
    "Mosque": "Visitors are welcome to view the architecture from outside prayer times.",
    "Temple": "It offers visitors a glimpse into local religious life.",
    "Ruins": "The site offers insight into the region's historical past for visitors interested in archaeology.",
    "Building": "It stands as a marker of the area's local history.",
    "Cascade": "Visitors can enjoy the surrounding scenery and, where safe, a refreshing dip.",
    "Sandy Beach": "It's a good spot for swimming, relaxing, and watching the sunset.",
    "National Park": "Visitors can look out for local wildlife while exploring the area.",
    "Viewpoint": "It offers a scenic spot to pause and take in the surrounding views.",
    "Museum": "It offers visitors a chance to learn more about local history and culture.",
    "Adventure Park": "It's a popular spot for a family day out.",
    "Tea Estate": "Visitors can enjoy views of the tea plantations and surrounding hills.",
    "Other": "It's worth a stop for travelers exploring the area.",
}


def generate_description_template(name, category_id, district):
    blurb = CATEGORY_BLURB.get(category_id, "a point of interest")
    opener = random.choice(OPENER_TEMPLATES).format(name=name, blurb=blurb, district=district)
    closer = CLOSER_TEMPLATES.get(category_id, CLOSER_TEMPLATES["Other"])
    return f"{opener} {closer}"


print(generate_description_template("Example Temple", "Buddhist Temple", "Kandy"))

## 10. LLM description generation (with template fallback + retry)

Prompted to use **only** the facts we actually have (name, category, district)
and explicitly told not to invent history, opening hours, prices, or other
details OSM didn't provide - same "extract/generate only from given facts,
never invent" discipline as the history notebook's extraction prompt. One or
two natural sentences, matching the length of the fallback template output.

Falls back to `generate_description_template` if the model isn't loaded, if
generation errors, or if the output fails basic sanity checks (empty, too
long, doesn't mention the place name).

In [ ]:
DESC_SYSTEM_PROMPT = (
    "You write short, natural-sounding one-to-two-sentence descriptions of "
    "Sri Lankan tourist places for a travel app. Use ONLY the facts given to "
    "you (name, category, district) - never invent history, dates, prices, "
    "opening hours, or other details you were not given. Do not use markdown. "
    "Reply with only the description text, no preamble."
)


def build_desc_prompt(name, category_id, district):
    blurb = CATEGORY_BLURB.get(category_id, "a point of interest")
    return (
        f'Place name: "{name}"\n'
        f"Category: {category_id} ({blurb})\n"
        f"District: {district}, Sri Lanka\n\n"
        "Write a one-to-two sentence description of this place for a travel app."
    )


def llm_generate(name, category_id, district, max_new_tokens=100, temperature=0.6):
    messages = [
        {"role": "system", "content": DESC_SYSTEM_PROMPT},
        {"role": "user", "content": build_desc_prompt(name, category_id, district)},
    ]
    inputs = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(llm_model.device)
    with torch.no_grad():
        out = llm_model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    text = llm_tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    return text.strip()


def is_valid_description(text, name):
    if not text or len(text) < 20 or len(text) > 600:
        return False
    # Loosely check the place is actually mentioned (first significant word of the name).
    first_word = re.sub(r"[^a-zA-Z]", "", name.split()[0]) if name.split() else ""
    if first_word and len(first_word) > 2 and first_word.lower() not in text.lower():
        return False
    return True


def generate_description(name, category_id, district, retries=2):
    if llm_ready:
        for _ in range(retries + 1):
            try:
                text = llm_generate(name, category_id, district)
            except Exception:
                break
            if is_valid_description(text, name):
                return text
    return generate_description_template(name, category_id, district)


if llm_ready:
    print(generate_description("Aluvihara Rock Temple", "Buddhist Temple", "Matale"))
else:
    print("LLM not active - generate_description() will use the template fallback.")
    print(generate_description("Aluvihara Rock Temple", "Buddhist Temple", "Matale"))

## 11. Default field values per category

OSM doesn't provide visitor-logistics fields (opening hours, safety level,
budget, etc). These are conservative, category-appropriate defaults - not
verified facts. Easy to spot for hand-review later since every record's `id`
is prefixed `pl_osm_`.

In [ ]:
DEFAULT_FIELDS = {
    "opening_hours": "Check locally",
    "mobile_signal": "Moderate",
    "road_condition": "Paved",
    "tourist_popularity": "Low",
    "family_friendly": "yes",
    "budget_category": "Free",
    "ticket_price": 0,
    "parking_avail": "unknown",
    "toilets": "unknown",
    "food_nearby": "unknown",
    "wheelchair_access": "unknown",
    "camping_allowed": "no",
    "safety_level": "Safe",
    "wildlife_hazard": "None",
    "guide_required": "no",
    "rain_sensitivity": "Caution",
    "monsoon_note": "Avoid May-Sept",
    "best_time_to_visit": "Dec to Mar",
    "Height_m": 0.0,
    "Length_km": 0.0,
    "Surfing": "no",
}

CATEGORY_OVERRIDES = {
    "National Park": {"wildlife_hazard": "Present - stay in vehicle/with guide", "guide_required": "yes"},
    "Sandy Beach": {"budget_category": "Free"},
    "Museum": {"budget_category": "Low"},
    "Tea Estate": {"budget_category": "Low"},
}

print("Default field templates ready.")

## 11b. Skip the LLM to save GPU quota (optional)

Run this cell **only if you're intentionally skipping Section 8's LLM load**
(e.g. to save GPU-hours quota) and going straight to Section 12. It forces
`generate_description()` to use the rule-based template fallback for every
place instead of raising a `NameError` for `llm_ready` not being defined.

Skip this cell entirely if you did run Section 8 normally - don't overwrite
a working `llm_ready`/`llm_model`/`llm_tokenizer` set up there.


In [ ]:
llm_ready = False
llm_model = None
llm_tokenizer = None
print("LLM skipped by choice - all descriptions will use the rule-based template fallback.")

## 12. Convert OSM elements -> TripMe schema records (resumable)

This is the slow step (one LLM call per place), so it's checkpointed:
each record is written to `osm_candidate_records.jsonl` as soon as it's built,
keyed by its OSM `(type, id)`. Re-running this cell after a disconnect skips
every element that already has a saved record instead of regenerating its
description from scratch.

In [ ]:
CANDIDATE_RECORDS_FILE = OUTPUT_DIR / "osm_candidate_records.jsonl"


def osm_center(el):
    if el["type"] == "node":
        return el.get("lat"), el.get("lon")
    center = el.get("center") or {}
    return center.get("lat"), center.get("lon")


def make_record(el):
    tags = el.get("tags", {})
    name = (tags.get("name:en") or tags.get("name") or "").strip()
    if not name:
        return None

    lat, lng = osm_center(el)
    if lat is None or lng is None:
        return None

    district = OSM_DISTRICT_ALIASES.get(el["_district"], el["_district"])
    category_id = el["_category_id"]

    try:
        description = generate_description(name, category_id, district)
    except Exception as e:
        # A single record's LLM call should never lose the rest of the batch -
        # fall back to the template and keep going.
        print(f"  description generation crashed for '{name}' ({e}) - using template.")
        description = generate_description_template(name, category_id, district)

    record = {
        "id": f"pl_osm_{uuid.uuid4().hex[:10]}",
        "name": name,
        "description": description,
        "district_id": district,
        "province_id": DISTRICT_PROVINCE[district],
        "category_id": category_id,
        "lat": round(float(lat), 6),
        "lng": round(float(lng), 6),
        "activities": el["_activities"],
        "_osm_type": el["type"],
        "_osm_id": el["id"],
        "_source_global_id": str(uuid.uuid4()),
    }
    defaults = dict(DEFAULT_FIELDS)
    defaults.update(CATEGORY_OVERRIDES.get(category_id, {}))
    for k, v in defaults.items():
        record[k] = v
    record["_output_file"] = el["_output_file"]
    return record


# Load whatever was already generated in a previous (possibly interrupted) run.
candidate_records = []
done_osm_keys = set()
if CANDIDATE_RECORDS_FILE.exists():
    with open(CANDIDATE_RECORDS_FILE, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                # A line got cut off mid-write (e.g. the session died exactly
                # while flushing) - skip just that line rather than losing
                # every record that came before it.
                print(f"  skipping unreadable line {line_no} in {CANDIDATE_RECORDS_FILE.name}")
                continue
            candidate_records.append(rec)
            done_osm_keys.add((rec["_osm_type"], rec["_osm_id"]))

print(f"Resuming: {len(candidate_records)} records already generated in a previous run.")

# De-dupe raw OSM elements by their own (type, id) first - the same place can
# match more than one category filter (e.g. a temple tagged both
# amenity=place_of_worship and tourism=attraction).
elements_to_process = []
seen_osm_ids = set(done_osm_keys)
for el in raw_osm_elements:
    osm_key = (el["type"], el["id"])
    if osm_key in seen_osm_ids:
        continue
    seen_osm_ids.add(osm_key)
    elements_to_process.append(el)

print(f"{len(elements_to_process)} new elements to generate descriptions for "
      f"(skipping {len(raw_osm_elements) - len(elements_to_process) - len(done_osm_keys)} "
      f"same-run duplicates and {len(done_osm_keys)} already-done from a prior run).")

with open(CANDIDATE_RECORDS_FILE, "a", encoding="utf-8") as out_f:
    for el in tqdm(elements_to_process, desc="Generating descriptions"):
        try:
            rec = make_record(el)
        except Exception as e:
            print(f"  skipping element {el.get('type')}/{el.get('id')} - unexpected error: {e}")
            continue
        if rec is None:
            continue
        candidate_records.append(rec)
        out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        out_f.flush()

print(f"\n{len(candidate_records)} total candidate records ready "
      f"(from {len(raw_osm_elements)} raw OSM elements).")

## 13. Deduplicate against the existing dataset

A candidate is dropped if a place with a similar normalized name already
exists **within ~300m** of it - catches the same site re-tagged with slightly
different spelling/casing, without discarding genuinely distinct places that
happen to share a common name (e.g. many "Purana Viharaya" temples).

In [ ]:
import math


def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def is_duplicate(record):
    candidates = existing_by_norm_name.get(norm_name(record["name"]), [])
    for existing in candidates:
        dist = haversine_m(record["lat"], record["lng"], existing["lat"], existing["lng"])
        if dist <= 300:
            return True
    return False


deduped_records = []
seen_in_run = set()
dropped_existing = 0
dropped_within_run = 0
for rec in candidate_records:
    if is_duplicate(rec):
        dropped_existing += 1
        continue
    run_key = (norm_name(rec["name"]), round(rec["lat"], 3), round(rec["lng"], 3))
    if run_key in seen_in_run:
        dropped_within_run += 1
        continue
    seen_in_run.add(run_key)
    deduped_records.append(rec)

print(f"Kept {len(deduped_records)} records.")
print(f"Dropped {dropped_existing} as duplicates of existing places.json entries.")
print(f"Dropped {dropped_within_run} as duplicates within this OSM pull itself.")

## 14. Save output — one `.jsonl` file per category

Each category's records go to their own file (`osm_buddhist_temple.jsonl`,
`osm_waterfall.jsonl`, `osm_beach.jsonl`, ...), pretty-printed with one record
per multi-line block, matching `data/raw/place_data.jsonl`'s format so
`01_merge_places.py`'s `.jsonl` loader (which decodes by scanning rather than
assuming one line per record) reads each of them correctly. The internal
`_output_file` tag used for routing is stripped before saving.

In [ ]:
from collections import Counter, defaultdict

by_file = defaultdict(list)
for rec in deduped_records:
    rec = dict(rec)
    output_file = rec.pop("_output_file")
    by_file[output_file].append(rec)

saved_files = []
for output_file, records in sorted(by_file.items()):
    out_path = OUTPUT_DIR / output_file
    with open(out_path, "w", encoding="utf-8") as f:
        for i, rec in enumerate(records):
            f.write(json.dumps(rec, indent=4, ensure_ascii=False))
            if i < len(records) - 1:
                f.write("\n")
            f.write("\n")
    saved_files.append((output_file, len(records)))
    print(f"Saved {len(records):4d} places -> {out_path}")

print()
print(f"Total: {sum(n for _, n in saved_files)} places across {len(saved_files)} category files.")
print()
print("By district (all categories combined):")
for d, c in Counter(r["district_id"] for r in deduped_records).most_common():
    print(f"  {d:15s} {c}")
print()
print("Next steps:")
print("  1. Download each osm_*.jsonl file listed above")
print("  2. Drop them all into data/raw/ in the project")
print("  3. Run: python scripts/01_merge_places.py")
print("  4. Check the new total place count printed by that script.")
print()
print("Attribution reminder: this data includes OpenStreetMap content -")
print('cite "© OpenStreetMap contributors" (ODbL) wherever this dataset is published.')

## 15. (Optional) Free GPU memory

In [ ]:
import gc

if llm_ready:
    del llm_model
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory released.")
else:
    print("No LLM was loaded - nothing to release.")